# Course 1 - NumPy I: The ND-Array

Live-coding notebook that mirrors the slide deck.
Concrete examples lifted from Aurelien Geron's NumPy tutorial.

**Sections**
1. The ND-Array (0:00-0:30)
2. Shape manipulation (0:30-1:00)
3. Indexing & slicing (1:00-1:30)

In [ ]:
import numpy as np
np.__version__


## 1. The ND-Array

### List vs. ndarray - same data, different machine

In [ ]:
py_list = [1, 2, 3, 4]
arr = np.array([1, 2, 3, 4])
print(py_list, type(py_list).__name__)
print(arr, type(arr).__name__, arr.dtype)


### Measure the difference

In [ ]:
%timeit sum(range(1_000_000))
%timeit np.arange(1_000_000).sum()


### Creating arrays - the recipes you'll use weekly

In [ ]:
print(np.zeros(5))
print(np.zeros((3, 4)))
print(np.ones((3, 4)))
print(np.full((3, 4), np.pi))


In [ ]:
print(np.arange(1, 5))
print(np.arange(1, 5, 0.5))
print(np.linspace(0, 5/3, 6))
print(np.eye(3))


### Randomness - use the modern Generator API

In [ ]:
rng = np.random.default_rng(seed=42)
print(rng.random((3, 4)))
print(rng.standard_normal((3, 4)))


### dtypes - promotion in action

In [ ]:
c = np.arange(1, 5)
print(c.dtype, c)        # int64

k1 = np.arange(0, 5, dtype=np.uint8)
k2 = k1 + np.array([5, 6, 7, 8, 9], dtype=np.int8)
print(k2.dtype, k2)      # int16 - promoted

k3 = k1 + 1.5
print(k3.dtype, k3)      # float64


### astype - convert dtype, get a new array

In [ ]:
g = np.arange(6, dtype=np.float64)
print(g.dtype, g)
h = g.astype(np.int32)
print(h.dtype, h)


### Inspect any array in six attributes

In [ ]:
a = np.zeros((3, 4))
print('shape   :', a.shape)
print('ndim    :', a.ndim)
print('size    :', a.size)
print('itemsize:', a.itemsize)
print('nbytes  :', a.nbytes)
print('dtype   :', a.dtype)


## 2. Shape manipulation

### Reshape, ravel, flatten

In [ ]:
g = np.arange(24)
g2 = g.reshape(6, 4)        # new view, same buffer
print(g2)
print(g2.ravel())            # back to 1-D (view if possible)
print(g2.flatten())          # always a copy


### Transpose

In [ ]:
m1 = np.arange(10).reshape(2, 5)
print(m1)
print(m1.T)                   # shorthand for .transpose()

t = np.arange(24).reshape(4, 2, 3)
print(t.transpose((1, 2, 0)).shape)   # (2, 3, 4)


### Add and remove dimensions

In [ ]:
v = np.array([1, 2, 3])
print(v.shape)
print(v[np.newaxis, :].shape)        # (1, 3) - row vector
print(v[:, np.newaxis].shape)        # (3, 1) - column vector
print(np.expand_dims(v, axis=0).shape)
print(np.squeeze(np.array([[[1, 2, 3]]])).shape)


### Memory contiguity - C vs F order (briefly)

In [ ]:
a = np.arange(12).reshape(3, 4)            # default: C-order (row-major)
print('C-contig?', a.flags['C_CONTIGUOUS'])
print('F-contig?', a.T.flags['F_CONTIGUOUS'])


## 3. Indexing & slicing

### Basic slicing - like Python, but multi-dim

In [ ]:
b = np.arange(48).reshape(4, 12)
print(b)
print('b[1, 2]   =', b[1, 2])
print('b[1, :]   =', b[1, :])
print('b[:, 1]   =', b[:, 1])
print('b[1:3,2:5]=\n', b[1:3, 2:5])
print('b[::-1]   =\n', b[::-1])


### Strided slicing

In [ ]:
a = np.arange(20).reshape(4, 5)
print(a)
print(a[1:3, ::2])     # rows 1-2, every other column


### Fancy (integer-array) indexing - pick whatever rows/cols you want

In [ ]:
print(b[(0, 2), 2:5])                       # rows 0 and 2, cols 2-4
print(b[:, (-1, 2, -1)])                     # cols last, 2, last (repeats!)
print(b[(-1, 2, -1, 2), (5, 9, 1, 9)])       # paired indices - 4 cells


### Boolean masks

In [ ]:
m = np.array([20, -5, 30, 40])
print(m[m < 25])             # [20 -5]

rows_on = np.array([True, False, True, False])
print(b[rows_on, :])         # keep rows 0 and 2

print(b[b % 3 == 1])         # every elem where x % 3 == 1


### np.where - pick from two arrays based on a mask

In [ ]:
x = np.arange(10)
print(np.where(x % 2 == 0, x, -1))   # keep evens, replace odds with -1


### Gotcha: slices are *views*, not copies

In [ ]:
a = np.arange(12).reshape(3, 4)
view = a[:2, :2]
view[0, 0] = -999
print(a)        # the source mutated too!


### Recap
* ndarray is a contiguous typed buffer - that is the entire speed story.
* Reshape and transpose change *strides*, not data.
* Slice with `:`, `,` and `::`; pick anything with fancy or boolean indexing.

# Course 2 - NumPy II: Vectorization & Beyond

Live-coding notebook that mirrors the slide deck.
Concrete examples lifted from Aurelien Geron's NumPy tutorial.

**Sections**
1. Vectorization & ufuncs (0:00-0:30)
2. Broadcasting (0:30-1:00)
3. Linear algebra & stacking (1:00-1:30)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)


## 1. Vectorization & ufuncs

### The loop you stop writing

In [ ]:
x = rng.normal(size=1_000_000)

def slow(x):
    out = np.empty_like(x)
    for i in range(len(x)):
        out[i] = x[i]**2 + 3*x[i]
    return out

def fast(x):
    return x**2 + 3*x

%timeit slow(x)
%timeit fast(x)


### Element-wise arithmetic on paired arrays

In [ ]:
a = np.array([14, 23, 32, 41])
b = np.array([5,  4,  3,  2])
print(a + b)
print(a - b)
print(a * b)
print(a / b)
print(np.maximum(a, b))
print(np.greater(a, b))


### Universal functions (ufuncs) - math everywhere

In [ ]:
print(np.sqrt(a))
print(np.exp(a))
print(np.sin(a))
print(np.log(a))
print(np.maximum(a, 0))     # ReLU in one line
print(np.clip(a, 0, 30))


### Reductions - and the axis mnemonic

In [ ]:
a = np.array([[-2.5, 3.1, 7], [10, 11, 12]])
print('mean all :', a.mean())
print('std all  :', a.std())
print('min, max :', a.min(), a.max())
print('argmax   :', a.argmax())


In [ ]:
c = np.arange(24).reshape(2, 3, 4)
print('sum axis 0 :', c.sum(axis=0).shape)   # collapse axis 0 -> (3, 4)
print('sum axis 1 :', c.sum(axis=1).shape)   # collapse axis 1 -> (2, 4)
print('sum (0, 2) :', c.sum(axis=(0, 2)).shape)


## 2. Broadcasting

### Scalar broadcast

In [ ]:
a = np.arange(6).reshape(2, 3)
print(a)
print(a + 100)            # scalar broadcasts to every cell


### Row vector across rows: (2,3) + (3,)

In [ ]:
print(a + np.array([100, 200, 300]))


### Column vector across columns: (2,3) + (2,1)

In [ ]:
print(a + np.array([[100], [200]]))


### Two compatible shapes: (3,1) + (1,4) -> (3,4)

In [ ]:
col = np.arange(3)[:, None]      # (3, 1)
row = np.arange(4)[None, :]      # (1, 4)
print(col + row)                 # (3, 4) outer-sum


### Standardize a data matrix in one line (broadcast + reduction)

In [ ]:
X = rng.normal(loc=5, scale=2, size=(200, 5))
Z = (X - X.mean(axis=0)) / X.std(axis=0)
print('column means ~0:', Z.mean(axis=0).round(4))
print('column stds  ~1:', Z.std(axis=0).round(4))


## 3. Linear algebra & stacking

### Matrix multiplication - `@` is the operator

In [ ]:
n1 = np.arange(10).reshape(2, 5)
n2 = np.arange(15).reshape(5, 3)
print(n1 @ n2)                # equivalent to n1.dot(n2)


### Norms, inverse, determinant

In [ ]:
m = np.array([[1,  2,  3],
              [5,  7, 11],
              [21, 29, 31]], dtype=float)
print('||m||_F =', np.linalg.norm(m))
print('det m  =', np.linalg.det(m))
print('inv m  =\n', np.linalg.inv(m).round(3))


### Solve a linear system - `Ax = b`

In [ ]:
A = np.array([[2, 6], [5, 3]], dtype=float)
b = np.array([6, -9], dtype=float)
x = np.linalg.solve(A, b)
print('x      =', x)
print('A @ x  =', A @ x, '   should equal', b)


### Eigendecomposition - one-liner

In [ ]:
S = np.array([[4, 1], [1, 3]], dtype=float)
w, V = np.linalg.eig(S)
print('eigvals:', w)
print('eigvecs:\n', V.round(3))


### Stack & split

In [ ]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])
print(np.concatenate([a, b], axis=0))   # vstack
print(np.concatenate([a, b], axis=1))   # hstack
print(np.stack([a, b]).shape)           # new axis -> (2, 2, 2)


In [ ]:
big = np.arange(12).reshape(3, 4)
left, right = np.split(big, 2, axis=1)
print(left)
print(right)


### Recap
* Replace numeric `for` loops with broadcasting + reductions.
* `@` for matmul, `np.linalg.solve` for `Ax = b`.
* Combine arrays with `concatenate` / `stack` / `split`.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

# Exercise 1 - Multiplication Table

Goal: practise array creation + broadcasting in one tiny puzzle.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)


**Task 1.** Build a `5 x 5` multiplication table where entry `(i, j)` equals `(i+1) * (j+1)`. Use `np.arange` and broadcasting - no loops.

In [ ]:
# your code here


**Task 2.** Print its shape, dtype, and `nbytes`.

In [ ]:
# your code here


**Task 3.** Extract the diagonal as a 1-D array using a single fancy-index expression.

In [ ]:
# your code here


### Exercise 1 - Solution

# Exercise 1 - Solutions

In [ ]:
import numpy as np


**Task 1.**

In [ ]:
row = np.arange(1, 6)
table = row[:, None] * row[None, :]
print(table)


**Task 2.**

In [ ]:
print('shape :', table.shape)
print('dtype :', table.dtype)
print('nbytes:', table.nbytes)


**Task 3.**

In [ ]:
ix = np.arange(5)
print(table[ix, ix])


---

## Exercise 2

# Exercise 2 - Masking divisibility

In [ ]:
import numpy as np


**Task 1.** Build a 1-D array `v` of the integers 1..30 inclusive.

In [ ]:
# your code here


**Task 2.** Print every value in `v` that is divisible by 3 using a boolean mask.

In [ ]:
# your code here


**Task 3.** Replace every value divisible by 3 in `v` with `0` (use `np.where` and assign to a new array `out`).

In [ ]:
# your code here


**Task 4.** How many values in `v` are divisible by both 3 *and* 5? Compute it with a vectorized expression.

In [ ]:
# your code here


### Exercise 2 - Solution

# Exercise 2 - Solutions

In [ ]:
import numpy as np


**Task 1.**

In [ ]:
v = np.arange(1, 31)
print(v)


**Task 2.**

In [ ]:
print(v[v % 3 == 0])


**Task 3.**

In [ ]:
out = np.where(v % 3 == 0, 0, v)
print(out)


**Task 4.**

In [ ]:
count = int(((v % 3 == 0) & (v % 5 == 0)).sum())
print(count)


---

## Exercise 3

# Exercise 3 - Slicing & Reshape

Treat a 2-D array as an image; every CV model sees this shape.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
M = rng.random((8, 4))
print(M.round(2))


**Task 1.** Extract every other row of `M` (rows 0, 2, 4, 6) using a slice.

In [ ]:
# your code here


**Task 2.** Reverse the column order of `M` (mirror left-right).

In [ ]:
# your code here


**Task 3.** Reshape `M` into a 4x8 array (no data copy required) and print its shape.

In [ ]:
# your code here


**Task 4.** Convert `M` (8x4) into a 3-D `(8, 4, 1)` array using `np.newaxis`. Print the shape.

In [ ]:
# your code here


### Exercise 3 - Solution

# Exercise 3 - Solutions

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
M = rng.random((8, 4))


**Task 1.**

In [ ]:
print(M[::2].shape)
print(M[::2].round(2))


**Task 2.**

In [ ]:
print(M[:, ::-1].round(2))


**Task 3.**

In [ ]:
R = M.reshape(4, 8)
print(R.shape)


**Task 4.**

In [ ]:
E = M[:, :, np.newaxis]
print(E.shape)


---

## Exercise 4

# Exercise 1 - Vectorize a pairwise distance loop

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
pts = rng.normal(size=(50, 2))


**Task 1.** Here is a naive Python loop that computes pairwise distances. Run it once to see the result.

In [ ]:
def naive(pts):
    n = len(pts)
    out = np.empty((n, n))
    for i in range(n):
        for j in range(n):
            out[i, j] = np.sqrt(((pts[i] - pts[j])**2).sum())
    return out

D_naive = naive(pts)
print(D_naive.shape)


**Task 2.** Write a vectorized version `fast(pts)` that produces the same matrix using broadcasting - no Python loops.

In [ ]:
# your code here


**Task 3.** Verify `np.allclose(naive(pts), fast(pts))`.

In [ ]:
# your code here


**Task 4.** Time both versions with `%timeit`.

In [ ]:
# your code here


### Exercise 4 - Solution

# Exercise 1 - Solutions

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
pts = rng.normal(size=(50, 2))

def naive(pts):
    n = len(pts)
    out = np.empty((n, n))
    for i in range(n):
        for j in range(n):
            out[i, j] = np.sqrt(((pts[i] - pts[j])**2).sum())
    return out


**Task 2.**

In [ ]:
def fast(pts):
    diff = pts[:, None, :] - pts[None, :, :]
    return np.sqrt((diff**2).sum(axis=-1))

print(fast(pts).shape)


**Task 3.**

In [ ]:
assert np.allclose(naive(pts), fast(pts))
print('match!')


**Task 4.**

In [ ]:
%timeit naive(pts)
%timeit fast(pts)


---

## Exercise 5

# Exercise 2 - Broadcast-normalize

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
X = rng.normal(loc=5, scale=2, size=(100, 3))


**Task 1.** Compute the per-column mean and per-column std of `X` (each a length-3 array).

In [ ]:
# your code here


**Task 2.** Use broadcasting to produce `Z` such that each column has mean 0 and std 1.

In [ ]:
# your code here


**Task 3.** Verify the column means and stds of `Z` numerically.

In [ ]:
# your code here


**Task 4.** Repeat normalisation but row-wise (each row has mean 0, std 1). What changes in the broadcasting?

In [ ]:
# your code here


### Exercise 5 - Solution

# Exercise 2 - Solutions

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
X = rng.normal(loc=5, scale=2, size=(100, 3))


**Task 1.**

In [ ]:
mu = X.mean(axis=0)
sd = X.std(axis=0)
print('mu:', mu)
print('sd:', sd)


**Task 2.**

In [ ]:
Z = (X - mu) / sd
print(Z.shape)


**Task 3.**

In [ ]:
print('Z col means ~ 0:', Z.mean(axis=0).round(4))
print('Z col stds  ~ 1:', Z.std(axis=0).round(4))


**Task 4.**

In [ ]:
mu_r = X.mean(axis=1, keepdims=True)
sd_r = X.std(axis=1, keepdims=True)
Zr = (X - mu_r) / sd_r
print('row means ~0:', Zr.mean(axis=1)[:5].round(4))
print('row stds  ~1:', Zr.std(axis=1)[:5].round(4))


---

## Exercise 6

# Exercise 3 - Solve a 3x3 system

In [ ]:
import numpy as np


Consider the system

    2x + y - z = 8
   -3x - y + 2z = -11
   -2x + y + 2z = -3

**Task 1.** Build matrix `A` and vector `b`.

In [ ]:
# your code here


**Task 2.** Solve with `np.linalg.solve`.

In [ ]:
# your code here


**Task 3.** Verify the solution by computing `A @ x` and comparing with `b` via `np.allclose`.

In [ ]:
# your code here


**Task 4 (stretch).** Solve again using `np.linalg.inv(A) @ b` and compare. Which is preferred and why?

In [ ]:
# your code here


### Exercise 6 - Solution

# Exercise 3 - Solutions

In [ ]:
import numpy as np


**Task 1.**

In [ ]:
A = np.array([[ 2,  1, -1],
              [-3, -1,  2],
              [-2,  1,  2]], dtype=float)
b = np.array([8, -11, -3], dtype=float)


**Task 2.**

In [ ]:
x = np.linalg.solve(A, b)
print('x =', x)


**Task 3.**

In [ ]:
assert np.allclose(A @ x, b)
print('Ax =', A @ x, '   b =', b)


**Task 4.**

In [ ]:
x_inv = np.linalg.inv(A) @ b
print('via inv:', x_inv)
print('match  :', np.allclose(x, x_inv))
# solve(A, b) is preferred: faster and more stable than forming inv(A).
